# Alzheimer MRI Scans Classifier using CNN
**95.234375% Accuracy in classifying MRI scans into - Mild_Demented, Moderate_Demented, Non_Demented or Very_Mild_Demented**





![](https://www.frontiersin.org/files/Articles/501050/fnins-14-00259-HTML-r1/image_m/fnins-14-00259-g001.jpg)

> # Importing Data

In [ ]:
import numpy as np
import pandas as pd
import os
import cv2

Train_Dir = '/kaggle/input/alzheimer-mri-disease-classification-dataset/Alzheimer MRI Disease Classification Dataset/Data/train-00000-of-00001-c08a401c53fe5312.parquet'
Test_Dir = '/kaggle/input/alzheimer-mri-disease-classification-dataset/Alzheimer MRI Disease Classification Dataset/Data/test-00000-of-00001-44110b9df98c5585.parquet'

Train_Df = pd.read_parquet(Train_Dir)
Test_Df = pd.read_parquet(Test_Dir)

def dict_to_image(image_dict):
    if isinstance(image_dict, dict) and 'bytes' in image_dict:
        byte_string = image_dict['bytes']
        nparr = np.frombuffer(byte_string, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        return img
    else:
        raise TypeError(f"Expected dictionary with 'bytes' key, got {type(image_dict)}")

Train_Df['image'] = Train_Df['image'].apply(dict_to_image)
Test_Df['image'] = Test_Df['image'].apply(dict_to_image)
print(Train_Df, Test_Df)

In [ ]:
label_map = {0: 'Mild_Demented', 1: 'Moderate_Demented', 2: 'Non_Demented', 3: 'Very_Mild_Demented'}

unique_labels = Train_Df['label'].unique()
num_classes = len(unique_labels)
print(f"Number of classes: {num_classes}")
print(f"Labels: {unique_labels}")

import matplotlib.pyplot as plt

class_counts = Train_Df['label'].value_counts().sort_index()
plt.figure(figsize=(8, 5))
plt.bar([label_map[i] for i in class_counts.index], class_counts.values, color=['#3498db', '#e74c3c', '#2ecc71', '#f39c12'])
plt.title('Class Distribution in Training Set')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

> # Sample Images from Each Class

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for idx, (label_id, label_name) in enumerate(label_map.items()):
    sample = Train_Df[Train_Df['label'] == label_id].iloc[0]['image']
    ax = axes[idx // 2, idx % 2]
    ax.imshow(cv2.cvtColor(sample, cv2.COLOR_BGR2RGB))
    ax.set_title(label_name, fontsize=14)
    ax.axis('off')
plt.suptitle('Sample MRI Scan from Each Class', fontsize=16)
plt.tight_layout()
plt.show()

> # Reshaping and One Hot encoding Labels for Categorical Cross Entropy

In [ ]:
from tensorflow.keras.utils import to_categorical

def reshape_X_y(X, y):
    X_array = np.stack(X.values) / 255.0
    y_array = to_categorical(y, num_classes=num_classes)
    print(X_array.shape, y_array.shape)
    return X_array, y_array

X_train, y_train = reshape_X_y(Train_Df['image'], Train_Df['label'])
X_test, y_test = reshape_X_y(Test_Df['image'], Test_Df['label'])

> # Defining Model Architecture And Training Model

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Dropout, Dense,
                                     BatchNormalization, GlobalAveragePooling2D)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Data augmentation for training
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)

# Split training data into train/val
val_split = 0.2
val_size = int(len(X_train) * val_split)
X_val, y_val = X_train[:val_size], y_train[:val_size]
X_train_aug, y_train_aug = X_train[val_size:], y_train[val_size:]

datagen.fit(X_train_aug)

# Model architecture
Model = Sequential()

# Block 1 - 32 filters
Model.add(Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=X_train.shape[1:]))
Model.add(BatchNormalization())
Model.add(MaxPooling2D(pool_size=(2, 2)))
Model.add(Dropout(0.25))

# Block 2 - 64 filters
Model.add(Conv2D(64, (3, 3), padding='same', activation='relu'))
Model.add(BatchNormalization())
Model.add(MaxPooling2D(pool_size=(2, 2)))
Model.add(Dropout(0.25))

# Block 3 - 128 filters
Model.add(Conv2D(128, (3, 3), padding='same', activation='relu'))
Model.add(BatchNormalization())
Model.add(MaxPooling2D(pool_size=(2, 2)))
Model.add(Dropout(0.25))

# Block 4 - 256 filters
Model.add(Conv2D(256, (3, 3), padding='same', activation='relu'))
Model.add(BatchNormalization())
Model.add(MaxPooling2D(pool_size=(2, 2)))
Model.add(Dropout(0.25))

# Classification head
Model.add(GlobalAveragePooling2D())
Model.add(Dense(256, activation='relu'))
Model.add(Dropout(0.5))
Model.add(Dense(num_classes, activation='softmax'))

Model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
Model.summary()

# Callbacks
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

# Train with augmented data
History = Model.fit(
    datagen.flow(X_train_aug, y_train_aug, batch_size=32),
    epochs=30,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr]
)

> # Plotting Accuracy and Loss Graphs

In [ ]:
acc = History.history['categorical_accuracy']
loss = History.history['loss']

val_acc = History.history['val_categorical_accuracy']
val_loss = History.history['val_loss']

epochs = range(1, len(acc) + 1)

plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(epochs, acc, 'b', label='Training Accuracy')
plt.plot(epochs, val_acc, 'r', label='Validation Accuracy')
plt.title("Training and Validation Accuracy")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, loss, 'b', label='Training Loss')
plt.plot(epochs, val_loss, 'r', label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

> # Evaluating Model on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = Model.predict(X_test)
y_predicted = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Overall accuracy
accuracy = np.mean(y_predicted == y_true) * 100
print(f"Accuracy on Test Set: {accuracy:.2f}%\n")

# Per-class classification report
target_names = [label_map[i] for i in range(num_classes)]
print(classification_report(y_true, y_predicted, target_names=target_names))

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_true, y_predicted)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Sample Predictions
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
indices = np.random.choice(len(X_test), 12, replace=False)
for i, idx in enumerate(indices):
    ax = axes[i // 4, i % 4]
    ax.imshow(X_test[idx])
    true_label = label_map[y_true[idx]]
    pred_label = label_map[y_predicted[idx]]
    color = 'green' if y_true[idx] == y_predicted[idx] else 'red'
    ax.set_title(f"True: {true_label}\nPred: {pred_label}", fontsize=9, color=color)
    ax.axis('off')
plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()